# 01 Data Acquisition and Audit

Audits the dated source log, historical cutoff, raw-layer policy, and automated quality results.
The bundled incident excerpt validates the schema and pipeline only. Full NHTSA archives are acquired from official URLs when network transfer is available.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
assert (ROOT / "data").exists(), "Execute with project root as the working directory."
pd.set_option("display.max_colwidth", 120)


In [2]:
sources=pd.read_csv(ROOT/"data/manually_compiled/source_log.csv",parse_dates=["publication_date","access_date"])
sources[["source_id","organization","publication_date","source_class","cutoff_valid"]].head(10)

,source_id,organization,publication_date,source_class,cutoff_valid
0,SRC001,NHTSA,2025-04-24,Primary,True
1,SRC002,NHTSA,2025-06-15,Primary,True
2,SRC003,NHTSA,2025-06-15,Primary,True
3,SRC004,NHTSA,2025-04-24,Primary,True
4,SRC005,California DMV,2025-03-01,Primary,True
5,SRC006,California DMV,2025-01-31,Primary,True
6,SRC007,California DMV,2024-02-02,Primary,True
7,SRC008,California DMV,2025-03-01,Primary,True
8,SRC009,Waymo,2025-05-01,Primary,True
9,SRC010,Waymo,2025-05-05,Primary,True


In [3]:
cutoff=pd.Timestamp("2025-06-30")
audit=pd.Series({
    "sources":len(sources),
    "primary_sources":int((sources.source_class=="Primary").sum()),
    "post_cutoff_sources":int((sources.publication_date>cutoff).sum()),
    "missing_urls":int(sources.source_url.isna().sum()),
    "duplicate_source_ids":int(sources.source_id.duplicated().sum())
})
audit

sources                 35
primary_sources         33
post_cutoff_sources      0
missing_urls             0
duplicate_source_ids     0
dtype: int64

In [4]:
raw=pd.read_csv(ROOT/"data/raw/nhtsa_ads_curated_schema_excerpt.csv")
qa=pd.read_csv(ROOT/"data/data_quality/qa_validation_result.csv")
print("Raw excerpt rows:",len(raw))
print("Automated checks:",qa.status.value_counts().to_dict())
raw[["Report ID","Reporting Entity","Incident Date","City","State","Highest Injury Severity Alleged"]]

Raw excerpt rows: 3
Automated checks: {'PASS': 20}


,Report ID,Reporting Entity,Incident Date,City,State,Highest Injury Severity Alleged
0,30270-11087,Waymo LLC,JUN-2025,Phoenix,AZ,No Injuries Reported
1,30270-11085,Waymo LLC,JUN-2025,Austin,TX,No Injuries Reported
2,30270-10919,Waymo LLC,MAY-2025,Los Angeles,CA,No Injuries Reported


**Interpretation:** The source log passes the historical cutoff check. The raw excerpt is intentionally not used to calculate a safety rate.